### [The Quantitative Analyst’s Toolkit: Essential Pandas for Financial Data](https://medium.com/@simplifiedzone/the-quantitative-analysts-toolkit-essential-pandas-for-financial-data-ch3-p2-0a0ec88d23bb)

In [1]:
import pandas as pd
import numpy as np

# Setup: Create a Date Range and Random Data
dates = pd.date_range(start='2022-01-01', end='2023-12-31', freq='D')
prices = np.random.normal(100, 10, len(dates)) # Random prices
df = pd.DataFrame({'Price': prices}, index=dates)

# Select all data from the year 2023
data_2023 = df.loc['2023']

# Select all data from January 2022
jan_data = df.loc['2022-01']

print(f"Total Rows: {len(df)}")
print(f"2023 Rows: {len(data_2023)}")

Total Rows: 730
2023 Rows: 365


In [2]:
# Create Sample Daily Data
dates = pd.date_range('2023-01-01', periods=100, freq='D')
df = pd.DataFrame({
    'Price': np.random.uniform(100, 110, 100),
    'Volume': np.random.randint(1000, 5000, 100)
}, index=dates)

# Resample to End-of-Month ('M')
monthly_data = df.resample('M').agg({
    'Price': 'last',  # Closing price of the month
    'Volume': 'sum'   # Total volume traded
})

display(monthly_data)

/tmp/ipython-input-3644852726.py:9: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_data = df.resample('M').agg({


,Price,Volume
2023-01-31,101.105448,99804
2023-02-28,101.125516,84496
2023-03-31,107.829929,87588
2023-04-30,104.147229,37141


In [3]:
df = pd.DataFrame({'Price': [100, 102, 101, 105]}, index=['Mon', 'Tue', 'Wed', 'Thu'])

# Create a column for Previous Close
df['Prev_Price'] = df['Price'].shift(1)

display(df)

,Price,Prev_Price
Mon,100,NaN
Tue,102,100.0
Wed,101,102.0
Thu,105,101.0


In [4]:
# Manual Calculation
df['Return_Manual'] = (df['Price'] - df['Prev_Price']) / df['Prev_Price']

# Automatic Calculation
df['Return_Auto'] = df['Price'].pct_change()

display(df[['Price', 'Return_Auto', 'Return_Manual']])

,Price,Return_Auto,Return_Manual
Mon,100,NaN,NaN
Tue,102,0.020000,0.020000
Wed,101,-0.009804,-0.009804
Thu,105,0.039604,0.039604


In [5]:
# Create Sample Data: 100 days of prices
dates = pd.date_range('2023-01-01', periods=100)
prices = np.linspace(100, 150, 100) + np.random.normal(0, 2, 100) # Trend + Noise
df = pd.DataFrame({'Price': prices}, index=dates)

# Calculate a 20-Day Simple Moving Average
df['SMA_20'] = df['Price'].rolling(window=20).mean()

# We use NumPy's log function
df['Log_Return'] = np.log(df['Price'] / df['Price'].shift(1))

df.resample('W').max()
df['Monthly_Return_Proxy'] = df['Price'].pct_change(periods=20)

display(df.tail())

,Price,SMA_20,Log_Return,Monthly_Return_Proxy
2023-04-06,148.911558,143.060219,0.009633,0.076018
2023-04-07,150.457639,143.779358,0.010329,0.105698
2023-04-08,148.636737,144.421444,-0.012176,0.094567
2023-04-09,147.596662,144.660358,-0.007022,0.033457
2023-04-10,149.879614,144.993070,0.015349,0.046460


In [6]:
# Calculate 20-Day Rolling Standard Deviation
df['Rolling_Vol'] = df['Price'].rolling(window=20).std()

display(df.tail())

,Price,SMA_20,Log_Return,Monthly_Return_Proxy,Rolling_Vol
2023-04-06,148.911558,143.060219,0.009633,0.076018,3.663963
2023-04-07,150.457639,143.779358,0.010329,0.105698,3.632102
2023-04-08,148.636737,144.421444,-0.012176,0.094567,3.262627
2023-04-09,147.596662,144.660358,-0.007022,0.033457,3.313613
2023-04-10,149.879614,144.993070,0.015349,0.046460,3.491253


In [7]:
data = {
    'Ticker': ['AAPL', 'MSFT', 'XOM', 'CVX'],
    'Sector': ['Tech', 'Tech', 'Energy', 'Energy'],
    'Return': [0.15, 0.12, 0.05, 0.08]
}
df_port = pd.DataFrame(data)

# Calculate Average Return per Sector
sector_performance = df_port.groupby('Sector')['Return'].mean()

display(sector_performance)

,Return
Sector,
Energy,0.065
Tech,0.135


In [8]:
np.random.seed(42)
dates = pd.date_range(start='2020-01-01', periods=500, freq='B')
# Simulate a stock that generally goes up but has noise
price_noise = np.random.normal(0, 1, 500).cumsum()
prices = 100 + price_noise + np.linspace(0, 50, 500)

df = pd.DataFrame({'Price': prices}, index=dates)

# 1. Calculate 50-day SMA
df['SMA_50'] = df['Price'].rolling(window=50).mean()

# 2. Generate Position
# If Price > SMA, we want to be Long (1). Else Neutral (0).
# We using boolean mapping: True becomes 1, False becomes 0.
df['Position'] = np.where(df['Price'] > df['SMA_50'], 1, 0)

# 3. Shift the Position
# CRITICAL: We calculate the signal based on Today's Close,
# but we can only trade Tomorrow. So we shift position by 1 day.
df['Position'] = df['Position'].shift(1)

# Calculate Daily Market Returns
df['Market_Return'] = df['Price'].pct_change()

# Calculate Strategy Returns
# If Position was 1, we get the Market Return.
# If Position was 0, we get 0.
df['Strategy_Return'] = df['Market_Return'] * df['Position']

# Simple Cumulative Sum for quick comparison
df['Buy_Hold_Cum'] = df['Market_Return'].cumsum()
df['Strategy_Cum'] = df['Strategy_Return'].cumsum()

print("--- Performance Summary ---")
print(f"Buy & Hold Total Return: {round(df['Buy_Hold_Cum'].iloc[-1] * 100, 2)}%")
print(f"Strategy Total Return:   {round(df['Strategy_Cum'].iloc[-1] * 100, 2)}%")

# Check the last few rows
display(df.tail())

--- Performance Summary ---
Buy & Hold Total Return: 43.99%
Strategy Total Return:   36.75%


,Price,SMA_50,Position,Market_Return,Strategy_Return,Buy_Hold_Cum,Strategy_Cum
2021-11-24,156.504199,152.085523,1.0,0.004100,0.004100,0.459710,0.387309
2021-11-25,155.567153,152.121174,1.0,-0.005987,-0.005987,0.453723,0.381322
2021-11-26,155.477014,152.171817,1.0,-0.000579,-0.000579,0.453143,0.380742
2021-11-29,154.701597,152.215224,1.0,-0.004987,-0.004987,0.448156,0.375755
2021-11-30,153.418997,152.252160,1.0,-0.008291,-0.008291,0.439865,0.367464
